# 6 — Optional Cleanup

⚠️ **Permanently deletes AWS resources. Read each cell before running.**

This notebook tears down everything created by notebooks 1–4 in the **safe order** the AWS APIs require:

| Step | What it removes | Owned by |
|------|-----------------|----------|
| 1 | SageMaker endpoint, endpoint config, model | Notebook 2 (out-of-band) |
| 2 | Drift-monitor Lambda, EventBridge rule, SNS topic, CloudWatch dashboard + alarms | `scripts/deploy_lambda_container.sh` + `create_cloudwatch_monitoring.py` (out-of-band) |
| 3 | CloudFormation stack — SageMaker domain, monitoring-writer Lambda, IAM roles, SQS, S3 buckets, Athena DB | CFN |

Both shell scripts default to **dry-run** — they list what would be deleted but do nothing. Switch to `--execute` only when the dry-run output looks right.

[nbviewer](https://nbviewer.org/github/aws-samples/mlops-best-practices-on-aws/blob/main/sagemaker-automated-drift-and-trend-monitoring/notebooks/6_optional_cleanup.ipynb)

## Setup

In [ ]:
import sys
import os
import subprocess
from pathlib import Path
import boto3

# Find project root
project_root = Path.cwd()
while not (project_root / '.env').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

REGION = os.getenv('AWS_DEFAULT_REGION', 'us-west-2')
ENDPOINT_NAME = os.getenv('ENDPOINT_NAME', 'fraud-detection-endpoint')

print(f'Region:        {REGION}')
print(f'Endpoint name: {ENDPOINT_NAME}')
print(f'Project root:  {project_root}')

---
## Step 1 — SageMaker endpoint + config + model

Notebook 2 creates these via `boto3` (outside CFN), so CFN won't clean them up. We do this first because endpoints are billed per-hour for instance time.

The cell below **only lists** matching resources. To delete, uncomment the marked lines.

In [ ]:
sm = boto3.client('sagemaker', region_name=REGION)

def safe(call, label):
    try:
        return call()
    except Exception as e:
        print(f'  (skip {label}: {e.__class__.__name__})')
        return None

print('Endpoint:')
ep = safe(lambda: sm.describe_endpoint(EndpointName=ENDPOINT_NAME), 'endpoint')
if ep:
    print(f'  - {ENDPOINT_NAME}  status={ep["EndpointStatus"]}  config={ep["EndpointConfigName"]}')
else:
    print(f'  (no endpoint named {ENDPOINT_NAME})')

print('\nEndpoint configs (matching "fraud"):')
configs = sm.list_endpoint_configs(MaxResults=100).get('EndpointConfigs', [])
fraud_configs = [c['EndpointConfigName'] for c in configs if 'fraud' in c['EndpointConfigName'].lower()]
for c in fraud_configs:
    print(f'  - {c}')
if not fraud_configs:
    print('  (none)')

print('\nModels (matching "fraud"):')
models = sm.list_models(MaxResults=100).get('Models', [])
fraud_models = [m['ModelName'] for m in models if 'fraud' in m['ModelName'].lower()]
for m in fraud_models:
    print(f'  - {m}')
if not fraud_models:
    print('  (none)')

# ── To delete: uncomment the block below ─────────────────────────────────────
# if ep:
#     sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
#     print(f'✓ deleted endpoint {ENDPOINT_NAME}')
# for c in fraud_configs:
#     sm.delete_endpoint_config(EndpointConfigName=c)
#     print(f'✓ deleted endpoint config {c}')
# for m in fraud_models:
#     sm.delete_model(ModelName=m)
#     print(f'✓ deleted model {m}')

---
## Step 2 — Out-of-band drift-monitor resources

`scripts/delete_infrastructure.sh` tears down what notebook 3 deploys outside CFN: drift-monitor Lambda, EventBridge schedule, SNS topic + subscriptions, CloudWatch dashboard + 5 alarms.

### 2.1 — Dry-run preview (always safe)

In [ ]:
subprocess.run(['bash', 'scripts/delete_infrastructure.sh'], cwd=project_root)

### 2.2 — Execute (uncomment to actually delete)

Pass `--ecr` if you also want to delete the ECR repo (otherwise the image is kept around for fast redeploy).

In [ ]:
# subprocess.run(['bash', 'scripts/delete_infrastructure.sh', '--execute'], cwd=project_root)
# subprocess.run(['bash', 'scripts/delete_infrastructure.sh', '--execute', '--ecr'], cwd=project_root)

---
## Step 3 — CloudFormation stack

`cloudformation/delete_stack.sh` drains things CFN can't drain itself (S3 buckets, running JupyterLab apps, Lake Formation grants) then issues `delete-stack` and polls until done.

It **refuses to run** in execute mode if the drift-monitor Lambda is still present — that's the most common deletion failure, because the Lambda holds refs to the CFN-owned SQS queue, SNS topic, and IAM role.

### 3.1 — Dry-run preview

In [ ]:
subprocess.run(['bash', 'cloudformation/delete_stack.sh'], cwd=project_root)

### 3.2 — Execute (uncomment to actually delete)

Defaults to stack name `fraud-detection-monitoring`; pass another name as positional arg if needed.

In [ ]:
# subprocess.run(['bash', 'cloudformation/delete_stack.sh', '--execute'], cwd=project_root)

---
## Step 4 — What's left after delete-stack?

CFN deletes everything inside the stack but leaves these behind by default:

- **CloudWatch log groups** — `/aws/lambda/*`, `/aws/sagemaker/*`. AWS keeps them so you can debug failed deletes. Delete manually if you want a fully clean account:
  ```
  aws logs describe-log-groups --log-group-name-prefix /aws/lambda/fraud --region us-west-2 \
      --query 'logGroups[].logGroupName' --output text | \
      xargs -n1 aws logs delete-log-group --region us-west-2 --log-group-name
  ```

- **Athena query results** in `s3://<DataBucket>/athena-results/` — already gone because Step 3 emptied the bucket before deleting it.

- **ECR drift-monitor image** — kept by default in Step 2. Re-run Step 2.2 with `--ecr` to delete.

- **MLflow experiments** — stored inside the SageMaker MLflow tracking server, which IS in the stack, so this is gone too.